# 01 - Reading SPINE Output

Goal: open a reconstructed SPINE HDF5 file, inspect the high-level object hierarchy, and connect particle/interactions fields to analysis questions.

This notebook is the highest-priority hands-on material for the short session. It is adapted from the 2026 workshop HDF5 readback material, with inference moved out of scope.

## Runtime contract

Run inside `ghcr.io/deeplearnphysics/spine:latest`.

Set the sample in the first code cell:

```python
DETECTOR = "generic"
TAG = "tutorial"
GEOMETRY = None if DETECTOR == "generic" else DETECTOR
```

By convention, the notebooks read:

```python
HDF5_DATA_DIR / DETECTOR / f"{DETECTOR}_{TAG}_spine.hdf5"
```

and record the companion LArCV path:

```python
LARCV_DATA_DIR / DETECTOR / f"{DETECTOR}_{TAG}.root"
```

The HDF5 file should contain reconstructed particles/interactions and, for validation cells, truth objects.

In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

from spine.driver import Driver


In [ ]:

# Tutorial data convention at FNAL/EAF:
#   LARCV_DATA_DIR/DETECTOR/DETECTOR_TAG.root
#   HDF5_DATA_DIR/DETECTOR/DETECTOR_TAG_spine.hdf5
LARCV_DATA_DIR = Path("/exp/dune/data/users/drielsma/npc-ddas/larcv")
HDF5_DATA_DIR = Path("/exp/dune/data/users/drielsma/npc-ddas/reco")

# Edit these two lines to switch samples.
# Common detector examples: "icarus", "sbnd", "2x2", "nd-lar",
# "protodune-sp", "protodune-vd"
DETECTOR = "generic"
TAG = "tutorial"
GEOMETRY = None if DETECTOR == "generic" else DETECTOR

LARCV_PATH = LARCV_DATA_DIR / DETECTOR / f"{DETECTOR}_{TAG}.root"
DATA_PATH = HDF5_DATA_DIR / DETECTOR / f"{DETECTOR}_{TAG}_spine.hdf5"
print(DATA_PATH)


In [ ]:

CONFIG_CANDIDATES = [
    Path("../config/read_spine_hdf5.yaml"),
    Path("tutorials/config/read_spine_hdf5.yaml"),
]
CONFIG_PATH = next((path for path in CONFIG_CANDIDATES if path.exists()), None)

DATA_PATH = str(DATA_PATH)
if CONFIG_PATH is None:
    raise RuntimeError("Could not find tutorials/config/read_spine_hdf5.yaml")

cfg_text = CONFIG_PATH.read_text().replace("DATA_PATH", DATA_PATH)
cfg = yaml.safe_load(cfg_text)
if GEOMETRY:
    cfg["geo"] = {"detector": GEOMETRY}
driver = Driver(cfg)
print(f"Opened {DATA_PATH}")
print(f"Entries: {len(driver)}")
if GEOMETRY:
    print(f"Detector geometry: {GEOMETRY}")
print(f"Companion LArCV path: {LARCV_PATH}")


## Where to look things up

This tutorial deliberately pauses on small snippets. When a field or class is unfamiliar, use three complementary tools:

- Python introspection: `help(obj)`, `dir(obj)`, `obj.as_dict().keys()`
- The SPINE API browser: https://spine.readthedocs.io
- Source/config examples in `spine-prod`

The point is not to memorize every field. The point is to learn how to inspect the object model without guessing.

## Read One Entry

Start with exactly one event entry. The first thing to learn is the structure of the returned dictionary.

In [ ]:
ENTRY = 0
data = driver.process(entry=ENTRY)
list(data)

Now pull out the four high-level collections we will use. Pause here and check the counts before inspecting individual objects.

In [ ]:
reco_particles = data.get("reco_particles", [])
truth_particles = data.get("truth_particles", [])
reco_interactions = data.get("reco_interactions", [])
truth_interactions = data.get("truth_interactions", [])

In [ ]:
pd.DataFrame(
    {
        "collection": [
            "reco_particles",
            "truth_particles",
            "reco_interactions",
            "truth_interactions",
        ],
        "count": [
            len(reco_particles),
            len(truth_particles),
            len(reco_interactions),
            len(truth_interactions),
        ],
    }
)

## Inspect one object

SPINE objects expose Python attributes and usually an `as_dict()` method. Start by looking at one reconstructed particle and one reconstructed interaction.

Pause on the line `d = obj.as_dict()`: it is the fastest way to discover the analysis-facing fields saved in this file.

In [ ]:
particle = reco_particles[0]
interaction = reco_interactions[0]

type(particle), type(interaction)

In [ ]:
list(particle.as_dict())

In [ ]:
# Try one of these during the tutorial.
# help(particle)
# dir(particle)
# particle.as_dict().keys()

In [ ]:
def preview_value(value):
    if isinstance(value, np.ndarray):
        return f"array shape={value.shape} dtype={value.dtype}"
    return repr(value)[:120]

def compact_dict(obj, max_items=25):
    if hasattr(obj, "as_dict"):
        d = obj.as_dict()
    else:
        d = vars(obj)
    rows = [(key, preview_value(value)) for key, value in list(d.items())[:max_items]]
    return pd.DataFrame(rows, columns=["field", "value"])

compact_dict(particle)

In [ ]:
compact_dict(interaction)

## Use SPINE Labels

Do not hard-code PID or semantic-shape labels in analysis notebooks. Import the labels from SPINE so the notebook follows the installed package.

In [ ]:
try:
    from spine.constants import PID_LABELS, SHAPE_LABELS
except ImportError:
    from spine.utils.globals import PID_LABELS, SHAPE_LABELS

In [ ]:
display(pd.Series(SHAPE_LABELS, name="shape label").to_frame())
display(pd.Series(PID_LABELS, name="PID label").to_frame())

## Build a Particle Table

Pick a short list of fields first. This is the analysis contract you are choosing to rely on.

In [ ]:
PARTICLE_FIELDS = [
    "id",
    "interaction_id",
    "pid",
    "shape",
    "is_primary",
    "size",
    "depositions_sum",
]

In [ ]:
particle_df = pd.DataFrame(
    [{field: getattr(p, field, None) for field in PARTICLE_FIELDS} for p in reco_particles]
)
particle_df["pid_label"] = particle_df["pid"].map(PID_LABELS)
particle_df["shape_label"] = particle_df["shape"].map(SHAPE_LABELS)
particle_df

## Build an Interaction Table

Interactions group particles. The next table summarizes multiplicities and primary content without digging into every particle yet.

In [ ]:
def primary_particles(interaction):
    return [p for p in getattr(interaction, "particles", []) if getattr(p, "is_primary", False)]

In [ ]:
interaction_df = pd.DataFrame(
    [
        {
            "id": getattr(ia, "id", None),
            "nu_id": getattr(ia, "nu_id", None),
            "size": getattr(ia, "size", None),
            "vertex": getattr(ia, "vertex", None),
            "n_particles": len(getattr(ia, "particles", [])),
            "n_primary": len(primary_particles(ia)),
            "primary_pids": [getattr(p, "pid", None) for p in primary_particles(ia)],
            "topology": getattr(ia, "topology", None),
        }
        for ia in reco_interactions
    ]
)
interaction_df

## Live exercise

Pick one interaction and answer:

- Which primary particles does SPINE reconstruct?
- Is the vertex close to the visually obvious interaction point?
- Which fields would you trust for a first-pass analysis selection, and which require validation?

In [ ]:
from spine.vis.out import Drawer

drawer = Drawer(data)
fig = drawer.get("interactions", "id")
fig.show()

## Inference belongs in `spine-prod`

This short tutorial starts from HDF5 output. For real production, `spine-prod` should own the validated full-chain inference configs, model weights, campaign metadata, and file bookkeeping. That lets analysis notebooks stay focused on object interpretation and validation.

## Offline extensions

- Repeat the field inspection for a second detector sample and list which object fields are detector-dependent.
- Build a small field glossary for the 10 particle/interactions attributes your analysis will use.
- Compare the same event in this notebook and Spinal Tap, then record which table fields explain the visual topology.